# 第4章 pandas数据分析 · 课堂代码

> 本 notebook 与课件《Python金融数据分析 · 第4章 pandas数据分析》配套。
> 内容改编自《Python金融大数据分析（第2版）》第5章。

**使用说明**
- 点击单元格，按 `Shift + Enter` 运行；
- DataFrame 在 Notebook 中会以漂亮的表格形式显示。

新增部分为自编教学例子，参考 Wes McKinney《Python for Data Analysis》第3版的主题组织：[在线书](https://wesmckinney.com/book/)。先预测输出，再执行；练习答案可展开查看。

## 多周学习路线

本 Notebook 的第1—5节作为单元一；随后依次学习独立的 `04_2_Cleaning.ipynb`、`04_3_Wrangling.ipynb`、`04_4_GroupBy.ipynb`。最后回到本 Notebook 第6—9节复习金融时间序列、读写与风险分析。

每个补充 Notebook 都从自己的小数据开始。回到本文件时请从头执行以恢复变量。工作日频率 `B` 不是交易所日历；全部模拟行情仅用于语法演示。

## 路径准备

输入数据放在本章 `data/`，运行生成的文件放在 `outputs/`，Notebook放在 `notebooks/`。下面统一设置路径，Windows和macOS共用；可从本章notebooks目录、本章目录或课程根目录运行。

In [1]:
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    CHAPTER = cwd.parent
elif cwd.name == "Chapter4 - pandas数据分析":
    CHAPTER = cwd
else:
    CHAPTER = cwd / "Chapter4 - pandas数据分析"
assert CHAPTER.is_dir(), "请从本章notebooks目录、本章目录或课程根目录运行"
DATA = CHAPTER / "data"
OUTPUTS = CHAPTER / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


## 1. 初识 pandas

NumPy 数组没有行列标签，通常使用统一的数据类型，而真实行情有日期、代码、缺失值。pandas 给数组加上行列标签，成为"数据表"。

- **Series**：一列带索引的数据（如某股票的收盘价序列）
- **DataFrame**：一张带行列标签的表（如 多股票×多日期 的价格表）

In [2]:
import numpy as np
import pandas as pd        # 约定俗成，照抄
print(pd.__version__)

2.3.3


## 2. Series：带标签的一列数据

左边是**索引**（标签），右边是值。不指定索引时默认为 0,1,2,...

In [3]:
s = pd.Series([1500.35, 128.60, 55.20],
              index=["600519", "000858", "601318"])
s

600519    1500.35
000858     128.60
601318      55.20
dtype: float64

Series 支持 NumPy 式向量化运算与布尔筛选。

**关键概念：对齐** —— 两个 Series 运算时按**索引对齐**而不是按位置，顺序打乱也不会错。

In [4]:
print(s["600519"])               # 1500.35  按标签取值
print(s[["600519", "601318"]])   # 取多个
print(s * 1.05)                  # 所有值涨5%（向量化）
print(s[s > 100])                # 筛选出大于100的

1500.35
600519    1500.35
601318      55.20
dtype: float64
600519    1575.3675
000858     135.0300
601318      57.9600
dtype: float64
600519    1500.35
000858     128.60
dtype: float64


In [5]:
# 体会"按索引对齐"：两个顺序不同的 Series 相乘
weights = pd.Series([0.5, 0.3, 0.2], index=["601318", "600519", "000858"])
print(s * weights)     # 每个价格只与"自己代码"的权重相乘
print((s * weights).sum())   # 组合的加权价格

000858     25.720
600519    450.105
601318     27.600
dtype: float64
503.42499999999995


## 3. DataFrame：金融数据的"主战场"

字典的**键**变成列名。约定俗成的方向：**行=日期，列=资产**。

In [6]:
data = {"茅台": [1500.0, 1520.0, 1515.0],
        "五粮液": [128.0, 129.5, 127.0],
        "平安": [55.0, 54.5, 56.0]}
df = pd.DataFrame(data,
                  index=["2026-07-01", "2026-07-02", "2026-07-03"])
df

,茅台,五粮液,平安
2026-07-01,1500.0,128.0,55.0
2026-07-02,1520.0,129.5,54.5
2026-07-03,1515.0,127.0,56.0


拿到任何新数据，先把这些"体检动作"做一遍。`describe()` 相当于免费的初步报告。

In [7]:
print(df.shape)       # (3, 3)      行数、列数
print(df.columns)     # 列名
print(df.index)       # 行索引（这里是日期）

(3, 3)
Index(['茅台', '五粮液', '平安'], dtype='object')
Index(['2026-07-01', '2026-07-02', '2026-07-03'], dtype='object')


In [8]:
df.head(2)            # 前2行（head()默认5行）

,茅台,五粮液,平安
2026-07-01,1500.0,128.0,55.0
2026-07-02,1520.0,129.5,54.5


In [9]:
df.describe()         # 每列的统计摘要：均值、标准差、分位数等

,茅台,五粮液,平安
count,3.000000,3.000000,3.000000
mean,1511.666667,128.166667,55.166667
std,10.408330,1.258306,0.763763
min,1500.000000,127.000000,54.500000
25%,1507.500000,127.500000,54.750000
50%,1515.000000,128.000000,55.000000
75%,1517.500000,128.750000,55.500000
max,1520.000000,129.500000,56.000000


### 3A 先说明一行代表什么

这张表每行代表一次学生成绩记录，学号是标识而非数量。金融表也可以每行代表“股票—日期”，不必总是每列一只股票。

In [10]:
students = pd.DataFrame({
    "学号": ["001", "002", "003", "004"],
    "姓名": ["小林", "小周", "小陈", "小许"],
    "班级": ["A", "A", "B", "B"],
    "成绩": [82, 95, 68, 88]})
students.info()
students.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   学号      4 non-null      object
 1   姓名      4 non-null      object
 2   班级      4 non-null      object
 3   成绩      4 non-null      int64 
dtypes: int64(1), object(3)
memory usage: 260.0+ bytes


,学号,姓名,班级,成绩
0,001,小林,A,82
1,002,小周,A,95
2,003,小陈,B,68
3,004,小许,B,88


### 3B 一列与一列表：维度不同

单层方括号取一列得到 Series；列名列表得到 DataFrame。先观察 type 和 shape，再决定后面如何操作。

In [11]:
one_column = students["成绩"]
one_table = students[["成绩"]]
print(type(one_column), one_column.shape)
print(type(one_table), one_table.shape)
print(students.describe())

<class 'pandas.core.series.Series'> (4,)
<class 'pandas.core.frame.DataFrame'> (4, 1)
              成绩
count   4.000000
mean   83.250000
std    11.470978
min    68.000000
25%    78.500000
50%    85.000000
75%    89.750000
max    95.000000


## 4. 数据选取与筛选

- `df[...]` 里放列名：取**列**；
- 想按行取，用位置切片或 `loc/iloc`；
- **最常见的困惑**：`df["茅台"]` 可以，`df["2026-07-01"]` 却报错，因为 `[]` 默认针对列。

In [12]:
print(df["茅台"])            # 取一列 -> Series
df[["茅台", "平安"]]          # 取多列 -> DataFrame

2026-07-01    1500.0
2026-07-02    1520.0
2026-07-03    1515.0
Name: 茅台, dtype: float64


,茅台,平安
2026-07-01,1500.0,55.0
2026-07-02,1520.0,54.5
2026-07-03,1515.0,56.0


In [13]:
df[0:2]          # 前两行（位置切片，含头不含尾）

,茅台,五粮液,平安
2026-07-01,1500.0,128.0,55.0
2026-07-02,1520.0,129.5,54.5


**loc 按标签选；iloc 按位置选。**先想清楚"我要按名字找还是按位置找"，再选工具。

In [14]:
print(df.loc["2026-07-02"])              # 该行全部列
print(df.loc["2026-07-02", "茅台"])      # 指定行、指定列 -> 1520.0
print(df.loc[:, "茅台"])                 # 所有行、茅台列

茅台     1520.0
五粮液     129.5
平安       54.5
Name: 2026-07-02, dtype: float64
1520.0
2026-07-01    1500.0
2026-07-02    1520.0
2026-07-03    1515.0
Name: 茅台, dtype: float64


In [15]:
print(df.iloc[0])             # 第0行（按位置）
print(df.iloc[0:2, 1])        # 前2行、第1列
print(df.iloc[-1])            # 最后一行

茅台     1500.0
五粮液     128.0
平安       55.0
Name: 2026-07-01, dtype: float64
2026-07-01    128.0
2026-07-02    129.5
Name: 五粮液, dtype: float64
茅台     1515.0
五粮液     127.0
平安       56.0
Name: 2026-07-03, dtype: float64


条件筛选与 NumPy 布尔索引完全一致：多条件组合用 `&`、`|`，每个条件加括号。

**课堂练习**：用一行代码选出"五粮液价格低于128"的所有交易日。

In [16]:
df[df["茅台"] > 1510]       # 茅台价格高于1510的日子

,茅台,五粮液,平安
2026-07-02,1520.0,129.5,54.5
2026-07-03,1515.0,127.0,56.0


In [17]:
df[(df["茅台"] > 1500) & (df["平安"] > 55)]

,茅台,五粮液,平安
2026-07-03,1515.0,127.0,56.0


In [18]:
# 课堂练习：在这里写下你的代码


<details><summary>参考答案（先自己动手！）</summary>

```python
df[df["五粮液"] < 128]
```
</details>

### 4A 标签切片与位置切片

学号设为索引后，loc 用学号找人，iloc 按当前位置找人。在此唯一且有序的索引上，loc 切片包括末端标签，iloc 不包括末端位置。

In [19]:
by_id = students.set_index("学号")
print(by_id.loc["001":"003", ["姓名", "成绩"]])
print(by_id.iloc[0:2, [0, 2]])
print(by_id.reset_index().columns.tolist())

     姓名  成绩
学号         
001  小林  82
002  小周  95
003  小陈  68
     姓名  成绩
学号         
001  小林  82
002  小周  95
['学号', '姓名', '班级', '成绩']


### 4B 条件筛选与安全赋值

用 loc 一次指定行与列进行赋值；不要在 df[条件][列名] 的临时结果上修改原表。保留原始成绩，另建调整列。

In [20]:
students["调整成绩"] = students["成绩"]
mask = students["班级"].eq("B")
students.loc[mask, "调整成绩"] = students.loc[mask, "成绩"] + 2
selected = students.loc[students["成绩"].ge(80),
                        ["姓名", "班级", "成绩"]].copy()
print(selected)

   姓名 班级  成绩
0  小林  A  82
1  小周  A  95
3  小许  B  88


**先动手**：筛选 A 班且成绩至少为 90 的学生，仅展示姓名与成绩。

In [21]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
students.loc[(students["班级"] == "A") &
             (students["成绩"] >= 90), ["姓名", "成绩"]]
# 小周，95分
```
</details>

### 4C 对齐：标签相同才相加

Series 运算按标签匹配。缺少对应项会得到缺失值；fill_value=0 只适合“未出现确实表示没有”的业务含义。

In [22]:
morning = pd.Series([3, 5], index=["茶", "咖啡"])
afternoon = pd.Series([2, 4], index=["咖啡", "果汁"])
print(morning + afternoon)
print(morning.add(afternoon, fill_value=0))

咖啡    7.0
果汁    NaN
茶     NaN
dtype: float64
咖啡    7.0
果汁    4.0
茶     3.0
dtype: float64


## 5. 排序与分组

排序返回**新表**，原表不变。时间序列数据到手先 `sort_index()`，保证日期升序。

In [23]:
df.sort_values("茅台", ascending=False)  # 按茅台价格降序

,茅台,五粮液,平安
2026-07-02,1520.0,129.5,54.5
2026-07-03,1515.0,127.0,56.0
2026-07-01,1500.0,128.0,55.0


In [24]:
df.sort_index()                          # 按行索引(日期)排序

,茅台,五粮液,平安
2026-07-01,1500.0,128.0,55.0
2026-07-02,1520.0,129.5,54.5
2026-07-03,1515.0,127.0,56.0


**groupby 三步走**：分组（groupby）→ 选列 → 聚合（mean/sum/max...）。回答"每个行业的平均市盈率"这类问题，一行搞定。

In [25]:
stocks = pd.DataFrame({
    "代码":  ["600519", "000858", "601318", "600036"],
    "行业":  ["白酒", "白酒", "保险", "银行"],
    "市盈率": [28.5, 18.2, 8.1, 6.3]})
stocks

,代码,行业,市盈率
0,600519,白酒,28.5
1,000858,白酒,18.2
2,601318,保险,8.1
3,600036,银行,6.3


In [26]:
stocks.groupby("行业")["市盈率"].mean()

行业
保险     8.10
白酒    23.35
银行     6.30
Name: 市盈率, dtype: float64

In [27]:
# 也可以一次算多个统计量
stocks.groupby("行业")["市盈率"].agg(["mean", "max", "count"])

,mean,max,count
行业,,,
保险,8.10,8.1,1
白酒,23.35,28.5,2
银行,6.30,6.3,1


## 6. 金融时间序列

### 6.1 把字符串变成日期

`pd.to_datetime()` 把字符串转成真正的日期类型；之后可以按年、月智能取值：`df.loc["2026-07"]` 直接取出2026年7月的数据。`date_range` 生成规则日期序列，`freq="B"` 表示工作日。

In [28]:
df.index = pd.to_datetime(df.index)
df.index       # DatetimeIndex

DatetimeIndex(['2026-07-01', '2026-07-02', '2026-07-03'], dtype='datetime64[ns]', freq=None)

In [29]:
pd.date_range("2026-01-01", periods=5, freq="B")
# 从1月1日起的5个工作日（B = Business day）

DatetimeIndex(['2026-01-01', '2026-01-02', '2026-01-05', '2026-01-06',
               '2026-01-07'],
              dtype='datetime64[ns]', freq='B')

### 6.2 收益率计算：pct_change 与 shift

- `pct_change()`：把第3章的手写公式封装成方法，整张表所有列**一次全算**；
- `shift(1)`：错位是金融计算的核心技巧——"昨日收益""信号与次日收益配对"都靠它；
- 第一行没有昨日数据，结果自动为 NaN。

In [30]:
df.pct_change(fill_method=None)        # 每列的日收益率：(今日-昨日)/昨日

,茅台,五粮液,平安
2026-07-01,NaN,NaN,NaN
2026-07-02,0.013333,0.011719,-0.009091
2026-07-03,-0.003289,-0.019305,0.027523


In [31]:
print(df.shift(1))     # 整表向下错位一行 -> 昨日价格
print(df.diff())       # 价格变动额（今日-昨日）

                茅台    五粮液    平安
2026-07-01     NaN    NaN   NaN
2026-07-02  1500.0  128.0  55.0
2026-07-03  1520.0  129.5  54.5


              茅台  五粮液   平安
2026-07-01   NaN  NaN  NaN
2026-07-02  20.0  1.5 -0.5
2026-07-03  -5.0 -2.5  1.5


### 6.3 重采样：日频转月频

`resample(频率)` 像"汇总报表"，后面必须接聚合方式（last/mean/sum）。常用频率：`"D"` 日、`"W"` 周、`"ME"` 月末、`"YE"` 年末。

In [32]:
df.resample("ME").last()    # 每月最后一个交易日的价格

,茅台,五粮液,平安
2026-07-31,1515.0,127.0,56.0


## 7. 缺失值处理

NaN 表示缺失：停牌、数据未披露、`pct_change` 的第一行。`ffill()`（前向填充）是沿用上次观测值的处理方式，是否适合须依据缺失原因判断。处理前先想：缺失代表什么经济含义？

In [33]:
s2 = pd.Series([1500.0, np.nan, 1515.0])

print(s2.isna())             # [False, True, False]  找出缺失
print(s2.dropna())           # 删掉缺失的行
print(s2.fillna(s2.mean()))  # 用均值填充
print(s2.ffill())            # 用前一个有效值填充（金融常用！）

0    False
1     True
2    False
dtype: bool
0    1500.0
2    1515.0
dtype: float64
0    1500.0
1    1507.5
2    1515.0
dtype: float64
0    1500.0
1    1500.0
2    1515.0
dtype: float64


## 8. 读写数据

真实数据多以 CSV/Excel 形式存在。读入时顺手完成"第一列当日期索引"：

```python
df = pd.read_csv(DATA / "eod_data.csv", index_col=0, parse_dates=True)
```

下面先把手头的表导出为 CSV，再读回来，体验完整的往返流程。

In [34]:
df.to_csv(OUTPUTS / "demo_prices.csv")                       # 导出
df2 = pd.read_csv(OUTPUTS / "demo_prices.csv",               # 读回
                  index_col=0,                     # 第0列作为索引
                  parse_dates=True)                # 解析为日期
print(type(df2.index))
df2

<class 'pandas.core.indexes.datetimes.DatetimeIndex'>

,茅台,五粮液,平安
2026-07-01,1500.0,128.0,55.0
2026-07-02,1520.0,129.5,54.5
2026-07-03,1515.0,127.0,56.0


## 9. 综合案例：三股风险分析

构造 250 个交易日的三只股票价格，完成"收益率—风险—相关性"分析。**一个方法作用于所有列**，全程没有循环。

In [35]:
rng = np.random.default_rng(42)
dates = pd.date_range("2026-01-05", periods=250, freq="B")

# 模拟三只股票的日对数收益率，再还原为价格
rets = pd.DataFrame({
    "茅台": 0.0004 + 0.015 * rng.standard_normal(250),
    "平安": 0.0002 + 0.020 * rng.standard_normal(250),
    "招行": 0.0003 + 0.018 * rng.standard_normal(250)},
    index=dates)
prices = 100 * np.exp(rets.cumsum())
prices.tail(3)

,茅台,平安,招行
2026-12-16,89.651397,126.024709,67.261712
2026-12-17,93.681881,121.129685,66.833219
2026-12-18,92.086771,117.584889,65.787109


In [36]:
rets = prices.pct_change(fill_method=None).dropna()   # 日收益率

rets.describe().round(4)              # 统计摘要

,茅台,平安,招行
count,249.0000,249.0000,249.0000
mean,-0.0003,0.0009,-0.0016
std,0.0142,0.0196,0.0185
min,-0.0313,-0.0498,-0.0517
25%,-0.0105,-0.0116,-0.0152
50%,-0.0013,0.0020,-0.0023
75%,0.0084,0.0130,0.0104
max,0.0451,0.0535,0.0473


In [37]:
print("年化平均收益：")
print((rets.mean() * 252).round(4))

print("年化波动率：")
print((rets.std() * np.sqrt(252)).round(4))

年化平均收益：
茅台   -0.0633
平安    0.2195
招行   -0.4056
dtype: float64
年化波动率：
茅台    0.2250
平安    0.3112
招行    0.2935
dtype: float64


In [38]:
rets.corr().round(2)                  # 相关系数矩阵

,茅台,平安,招行
茅台,1.00,0.04,-0.01
平安,0.04,1.00,-0.05
招行,-0.01,-0.05,1.00


**课堂讨论**：相关系数接近 1 的两只股票，能互相分散风险吗？

## 10. 课后任务

1. 运行本 notebook，替换 `freq`、随机种子等参数观察输出；
2. 对案例数据：找出每只股票的最大单日跌幅及其日期（提示：`idxmin()` 返回最小值所在的索引）；
3. 选做：用 `resample("ME").last()` 得到月末价格，再算月度收益率。